# Lab | Agent & Vector store: Solution

An agent that routes between two knowledge bases: Arthur Schopenhauer's *The Wisdom of Life* (public domain, Project Gutenberg) and the Ruff FAQ.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-text-splitters<0.3" python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)
os.environ.setdefault("USER_AGENT", "ironhack-lab")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(OPENAI_API_KEY))

In [ ]:
from langchain.agents import AgentType, Tool, initialize_agent
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

## Knowledge base 1: Schopenhauer

In [ ]:
schopenhauer_path = "./datasets/schopenhauer_studies_in_pessimism.txt"

schop_documents = TextLoader(schopenhauer_path, encoding="utf-8").load()

# Drop the Project Gutenberg license header/footer so only the book is embedded
raw = schop_documents[0].page_content
start = raw.find("*** START OF")
end = raw.find("*** END OF")
if start != -1:
    raw = raw[raw.find("\n", start) + 1 : end if end != -1 else None]
schop_documents[0].page_content = raw

schop_texts = text_splitter.split_documents(schop_documents)
print(f"{len(schop_texts)} chunks")

schop_db = Chroma.from_documents(schop_texts, embeddings, collection_name="schopenhauer")
schopenhauer = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=schop_db.as_retriever()
)

## Knowledge base 2: Ruff FAQ

In [ ]:
ruff_docs = WebBaseLoader("https://beta.ruff.rs/docs/faq/").load()
ruff_texts = text_splitter.split_documents(ruff_docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

## Agent as reasoner

In [ ]:
schop_description = (
    "useful for when you need to answer questions about Arthur Schopenhauer's philosophy in "
    "The Wisdom of Life: happiness, personality, wealth, honour, solitude, aphorisms on life. "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)
ruff_description = (
    "useful for when you need to answer questions about ruff (a python linter). "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)

tools = [
    Tool(name="Schopenhauer QA System", func=schopenhauer.run, description=schop_description),
    Tool(name="Ruff QA System", func=ruff.run, description=ruff_description),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?"
)

In [ ]:
agent.invoke("Why use ruff over flake8?")

In [ ]:
agent.invoke(
    "Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?"
)

## Agent as router (`return_direct=True`)

In [ ]:
direct_tools = [
    Tool(name="Schopenhauer QA System", func=schopenhauer.run, description=schop_description, return_direct=True),
    Tool(name="Ruff QA System", func=ruff.run, description=ruff_description, return_direct=True),
]

direct_agent = initialize_agent(
    direct_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
direct_agent.invoke(
    "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?"
)

In [ ]:
direct_agent.invoke("Why use ruff over flake8?")

In [ ]:
direct_agent.invoke(
    "Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?"
)

## Reflection

- **Schopenhauer-only question:** the agent chose the *Schopenhauer QA System* in both modes and answered that *what one is* (personality) matters most for happiness.
- **Ruff-only question:** the agent chose the *Ruff QA System* in both modes. The tool descriptions were enough to route correctly.
- **Multi-hop question:** the reasoner agent called the Ruff tool and the Schopenhauer tool and combined both answers.
- **`return_direct=True`:** the agent returned the first tool's answer immediately, with no extra reasoning. That is faster for single-source questions, but on the multi-hop question it stopped after one tool and only covered the Ruff half. For questions needing several sources, the reasoner agent works better.